# 03. Spatiotemporal Cyclone Tracking with 2D ConvLSTM Sequences

In this notebook:
1. We feed an 8-frame sequential satellite time series ($T=8$ frames at 3-hour intervals) into the 2D ConvLSTM encoder.
2. We autoregressively decode 48-hour forward projection displacements $(\Delta lat, \Delta lon)$.
3. We generate and visualize the expanding Cone of Uncertainty.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src.models.prediction_convlstm import CycloneTrajectoryConvLSTM
from src.utils.gis_visualization import generate_uncertainty_cone_geojson

model = CycloneTrajectoryConvLSTM(in_channels=4, forecast_steps=8)
model.eval()

# 8 historical steps [Batch=1, T=8, C=4, H=128, W=128]
dummy_seq = torch.randn(1, 8, 4, 128, 128)
forecast = model.forecast_track_trajectory(dummy_seq, current_lat=15.0, current_lon=88.0, current_wind_kts=85.0)

print("Generated Forecast Track Points:")
for pt in forecast:
    print(f"  {pt['forecast_time_offset']}: Lat {pt['latitude']}°N, Lon {pt['longitude']}°E, Wind {pt['estimated_wind_kts']} kts, Cone ±{pt['uncertainty_radius_km']} km")

## 2. Trajectory & Uncertainty Cone Visualizer

In [ ]:
lats = [15.0] + [p['latitude'] for p in forecast]
lons = [88.0] + [p['longitude'] for p in forecast]
radii = [15.0] + [p['uncertainty_radius_km'] for p in forecast]

plt.figure(figsize=(9, 8))
# Plot trajectory line
plt.plot(lons, lats, marker='o', color='crimson', lw=2.5, label='Predicted Trajectory (ConvLSTM)')

# Plot uncertainty envelopes
for lon, lat, r in zip(lons, lats, radii):
    circle = plt.Circle((lon, lat), r / 111.0, color='pink', alpha=0.3, ec='red')
    plt.gca().add_patch(circle)

plt.title('ConvLSTM 48-Hour Cyclone Trajectory & Cone of Uncertainty')
plt.xlabel('Longitude (°E)')
plt.ylabel('Latitude (°N)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()